# 02 — Activation Patching Baseline on Controlled NLI

This notebook runs the **activation patching** baseline for our controlled
lexical-NLI causal-abstraction study.

For each base/source NLI pair, for each `(layer, component, token position)`
site, we:

1. tokenize base and source,
2. run the unwrapped model on each (the *clean* base and source runs),
3. wrap the model in a `pyvene.IntervenableModel` with a single
   `VanillaIntervention` at the requested site,
4. patch the source activation into the base run and read off the
   counterfactual logits,
5. record per-example logit-difference recovery **and** whether the
   patched argmax matches the high-level counterfactual label.

The result is a tidy `pandas.DataFrame` with one row per
`(example, layer, component, position)`, which we then pivot into
layer × position heatmaps (one per component) and save under
`outputs/patching_heatmaps/`.

> Tip: the notebook is sized to run on a laptop CPU in a couple of minutes
> with `distilgpt2`. If you have a GPU, just set `DEVICE = "cuda"` below.

## 1. Setup

In [ ]:
import os, sys, pathlib

# Lightweight setup only. No torch / transformers / pyvene imports here.
# This cell should finish almost immediately.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add the project root (the directory that contains `src/`) to sys.path so
# we can import the project package regardless of where Jupyter was started.
_HERE = pathlib.Path.cwd()
for _p in [_HERE, *_HERE.parents]:
    if (_p / 'src').is_dir():
        PROJECT_ROOT = _p
        break
else:
    raise RuntimeError('Could not locate the project root (no src/ found).')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

MODEL_NAME = 'distilgpt2'   # small, runs on CPU
N_EXAMPLES = 20
LAYERS     = [0, 2, 4]      # distilgpt2 has 6 transformer blocks
COMPONENTS = ('mlp_output', 'attention_input', 'block_output')

OUT_DIR = PROJECT_ROOT / 'outputs' / 'patching_heatmaps'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

PROJECT_ROOT = /Users/abhiram3001/Downloads/course project stuff


## 2. Load model + verbalizer

In [ ]:
# First heavy imports happen here: torch + HuggingFace transformers.
import torch

from src.utils import set_seed
from src.utils.seed import get_device
from src.models import load_causal_lm
from src.metrics import LabelVerbalizer

set_seed(0)
DEVICE = get_device()   # cuda if available, else cpu
print('DEVICE =', DEVICE)

tokenizer, model = load_causal_lm(MODEL_NAME, device=DEVICE)
n_layers = getattr(model.config, 'n_layer', getattr(model.config, 'num_hidden_layers', None))
print(f'Loaded {MODEL_NAME} on {DEVICE}; n_layers = {n_layers}')

verbalizer = LabelVerbalizer.from_tokenizer(
    tokenizer,
    {'entailment': ' yes', 'neutral': ' maybe', 'contradiction': ' no'},
)
print('verbalizer token ids =', verbalizer.token_ids)

## 3. Build 20 controlled counterfactual NLI pairs

We target the `lexical_relation` intermediate variable in the high-level
causal model. The patch site (in the surface prompt) defaults to the
*hypothesis word* token — see `_VARIABLE_TO_SOURCE_WORD` in
`src.data.counterfactual_pairs`. We also require the counterfactual label
to differ from the base label, so chance-level IIA is meaningfully below
1.0.

In [ ]:
import pandas as pd

from src.data import build_counterfactual_dataset, ID2LABEL

dataset = build_counterfactual_dataset(
    tokenizer,
    target_variable='lexical_relation',
    n_examples=N_EXAMPLES,
    seed=0,
    require_label_change=True,
)
examples = dataset.examples
print(f'Built {len(examples)} counterfactual examples.')

preview = pd.DataFrame([
    {
        'base_prompt':   ex.base.prompt,
        'source_prompt': ex.source.prompt,
        'base_label':    ID2LABEL[ex.base_label_id],
        'source_label':  ID2LABEL[ex.source_label_id],
        'cf_label':      ID2LABEL[ex.counterfactual_label_id],
        'patch_pos':     ex.intervention_pos,
    }
    for ex in examples[:8]
])
preview

## 4. Run the patching sweep

On `distilgpt2` (CPU) with 20 examples, 3 layers, 3 components, and ~10
positions per sequence this takes roughly a minute. On a GPU it's a few
seconds.

In [ ]:
# This is the first pyvene-heavy import. If this import is slow, it should be
# slow here rather than in the first setup cell.
from src.interventions import run_patching_sweep

df = run_patching_sweep(
    model,
    tokenizer,
    examples,
    layers=LAYERS,
    components=COMPONENTS,
    positions='all',
    metric='logit_recovery',
    device=DEVICE,
    verbalizer=verbalizer,
    batch_size=8,
    progress=True,
)
print('rows:', len(df), '  base_accuracy =', round(df.attrs['base_accuracy'], 3))
df.head()

In [ ]:
# Quick summary: mean recovery + mean IIA per (layer, component).
summary = (
    df.groupby(['component', 'layer'])
      .agg(recovery=('recovery', 'mean'), iia=('iia_correct', 'mean'))
      .round(3)
      .reset_index()
)
summary

## 5. Save heatmaps per component

We save two figures per component:

- `*_recovery.png`  — mean logit-diff recovery, diverging colormap centered at 0.
- `*_iia.png`       — mean IIA (interchange-intervention accuracy), `[0, 1]` viridis.

In [ ]:
from src.viz import save_patching_heatmap_from_df

saved = []
for comp in COMPONENTS:
    rec_path = OUT_DIR / f'{comp}_recovery.png'
    iia_path = OUT_DIR / f'{comp}_iia.png'
    saved.append(save_patching_heatmap_from_df(df, str(rec_path), value_col='recovery',    component=comp, annot=True))
    saved.append(save_patching_heatmap_from_df(df, str(iia_path), value_col='iia_correct', component=comp, annot=True))

for p in saved:
    print('saved', p)

## 6. Persist the raw sweep as CSV

Useful for downstream analyses (per-example breakdowns, statistical tests,
etc.) without re-running the sweep.

In [ ]:
csv_path = PROJECT_ROOT / 'outputs' / 'patching_sweep.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
print('wrote', csv_path)

## Next steps

- Scale up to all 6 layers and all default components (still cheap with
  small models).
- Swap in `gpt2` or `gpt2-medium` and compare the heatmaps.
- Move from vanilla patching to **DAS**: replace `VanillaIntervention`
  with `RotatedSpaceIntervention` and train the rotation on the
  counterfactual dataset. The peak `(layer, component)` cells from this
  baseline are the natural starting points to localise the DAS subspace.